# 1. STABILIZED LAG BIAS — PAIRED β REPLICATION — AeroSync-Mamba v18

v18 starts from the v16 reference notebook and uses the same paired design as v17. It changes one thing: the
**LACA lag-bias mechanism g(Δτ)** gets a three-part stabilization package.

**Why.** v17 found two problems:
- β = 1.0 did not reliably beat β = 0.0. The mean paired difference was **+0.0008**, the std of the paired
  differences was **0.0020**, and only **3 of 5** seeds favored β = 1.0.
- The size of the lag-bias term, max|β·g(Δτ)|, ranged from **0.23 to 0.68** across seeds, even though β was always
  1.0. So the strength of the mechanism depended on the seed, which may explain why its benefit was inconsistent.

**The stabilization package.** All three changes are applied together.
1. **LayerNorm on g's output, before β.** g produces one scalar per (query i, key j) pair, a (B, T_m, T_n) grid
   per LACA direction. There is no feature dimension larger than 1 to normalize over; a LayerNorm over a size-1
   dimension would output 0 for every input. So the normalization is taken **over each window's whole (T_m, T_n)
   grid**, which is every bias value that enters that window's attention matrix. It has no affine parameters, so
   β is the only source of scale. The subtracted mean is one constant per window. Softmax is unchanged by a
   constant, so the mean subtraction does not affect attention. The division by the std is what matters.
2. **Zero-initialization.** g's final Linear(32, 1) weight and bias start at exactly 0. So g(Δτ) = 0 for every Δτ
   and every seed at initialization. `nn.init.zeros_` draws no random numbers, so every other layer's initial
   weights, and the whole shuffle and dropout stream, are identical to v16 and v17 for the same seed.
3. **β warm-up.** β_epoch = β_target · min(epoch, 5) / 5. With β_target = 1.0 this gives 0.2, 0.4, 0.6, 0.8 and
   1.0 over epochs 1–5, then 1.0 for epochs 6–20. For β_target = 0.0 it stays 0.0 at every epoch.

**Expected divergence from v16.** The architecture has changed, so seed 42 / β = 1.0 will **not** reproduce v16's
0.9080 / 0.9048 baseline. This is intentional and is not a bug. What *is* still checked:
- the data fingerprints;
- the torch RNG state right after model construction (it must still match v16, which proves the changes drew no
  random numbers);
- the **β = 0.0 arm against v17**. At β = 0 none of the three changes reach the output, so each β = 0 run should
  reproduce v17's β = 0 run exactly. This check runs only if v17's CSV is available.

**Design.** The same 10 runs as v17 (β ∈ {0.0, 1.0} × seeds 42, 123, 2024, 7, 99), in back-to-back pairs, logged
to `beta_stabilized_paired_results.csv` after every run. Section 7 computes the same paired statistics as v17, a
side-by-side table against v17, and answers four separate questions: (a) lag-contribution spread,
(b) mean paired difference, (c) std of the paired differences, (d) win count.

| Decision | Value | Evidence status |
|---|---|---|
| Timestamps into HTT / LACA | Real per-window audio and IMU timestamps (not a synthetic `linspace`) | CORRECTNESS FIX (earlier versions passed an identical synthetic grid for every window) |
| Fusion backbone | Real `mamba_ssm.Mamba` selective-scan SSM; pure-PyTorch fallback only when no CUDA GPU is available (printed unmissably) | CORRECTNESS FIX (the earlier gated-conv block was not an SSM) |
| Audio bin selection | Strongest-N FFT bins by mean magnitude, computed once from **training** audio only, then frozen and reused for val/test | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): strongest-64 0.8736 vs first-64 0.8494, +0.0242 (bin-selection study) |
| Frequency budget N | 64 bins | **PROVISIONAL** (margin exceeded the old same-seed placeholder ±0.0005 but has not yet been verified against the true measured seed-to-seed std of 0.0160). v6: N=64 0.8966 vs N=48 0.8863, **+0.0103**, which is below 0.0160. The other budgets clear it (N=32 +0.0249, N=128 +0.0166) except N=96 (+0.0142). |
| Audio compression | `sqrt(|rFFT|)` | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): v4 sqrt 0.8966 vs log_db 0.8696 (+0.0270) and log1p 0.8649 (+0.0317) |
| Audio windowing | None; Hann windowing rejected | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): v4 no window 0.8649 vs Hann 0.8290 (+0.0359) |
| Audio normalization | None; z-score normalization rejected | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): v4 none 0.8966 vs z-score 0.8684 (+0.0282) |
| Audio time positions Ta | 64 per 2.56 s window | PROVISIONAL: v7 Ta=128 0.8967 vs Ta=64 0.8966 (−0.0001, a tie). The cheaper Ta=64 is kept. |
| Sensor input | Raw 6 channels (accel x/y/z, gyro x/y/z), Ts=128; derived magnitude features rejected | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): v8 raw6 0.8966 vs best derived set 0.8740 (+0.0226) |
| LACA temporal-bias scale β | 1.0, reached by a 5-epoch linear warm-up (v18) | **NOT ESTABLISHED** (v17 paired 5 seeds: mean diff +0.0008, std of diffs 0.0020, β=1 won 3/5). **Re-tested in this notebook on the stabilized mechanism.** |
| LACA lag-bias form g(Δτ) | mlp `Linear(1,32) → Tanh → Linear(32,1)`, 97 params. **v18:** final Linear zero-initialized, output LayerNorm over each window's (T_m, T_n) grid (0 params) | PROVISIONAL: v10 mlp vs rbf +0.0029, vs bucketed +0.0030 (below 0.0160). The v18 stabilization is under test. |
| LACA heads | 4 | PROVISIONAL: v11 h=1 was +0.0008 (one test window) and h=4 was kept. h=2/8/16 were 0.006–0.012 worse (below 0.0160). |
| Mamba fusion | 2 blocks, d_state=16, d_conv=2 (expand 2) | **PROVISIONAL** (margin exceeded the old same-seed placeholder ±0.0005 but has not yet been verified against the true measured seed-to-seed std of 0.0160). v12: 2 vs 4 blocks +0.0023, d_state 16 vs 8 +0.0172, d_conv 2 vs 4 +0.0059. |
| λ_align (InfoNCE) | 0.1 | PROVISIONAL: v13 removing L_align cost −0.0088, which is below 0.0160. λ=0.05 was −0.0271 and λ=0.2 was −0.0070. |
| λ_cons (KL consistency) | 0.1 | **PROVISIONAL** (margin exceeded the old same-seed placeholder ±0.0005 but has not yet been verified against the true measured seed-to-seed std of 0.0160). v13 grid: λ=0.2 +0.0008, λ=0.05 −0.0232. Removing L_cons cost −0.0165. |
| Ordinal-severity loss | Not used (rejected) | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): v14 best λ_ord>0 0.8859 vs none 0.9048 (−0.0189); adjacent-class errors rose 94 → 113 |
| Classification loss | Cross-entropy weighted by inverse **training-set** class frequency | CORRECTNESS FIX (class-imbalance handling) |
| Seeding | Python `random`, numpy, torch CPU + all CUDA devices, cuDNN deterministic; reset per seed at run start and right before model construction. The mission split has its own fixed seed (42), so every seed sees the same train/val/test windows. | CORRECTNESS FIX (reproducibility) |

## 2. Configuration constants & environment
Every fixed value lives in the first cell below. The second installs `mamba_ssm` and reports which fusion backbone is active.

In [1]:
import copy
import csv
import gc
import hashlib
import importlib.util
import json
import math
import os
import random
import subprocess
import sys
import time
import urllib.request
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# ==============================================================================
# FIXED CONFIGURATION — every finalized choice (see the summary table above)
# ==============================================================================
# --- Paired beta replication: beta x seed ---
SEEDS = [42, 123, 2024, 7, 99]           # same 5 seeds as v15/v16, in this order
BETA_VALUES = [0.0, 1.0]                 # per seed, in this order (0.0 = lag-bias ablation, 1.0 = finalized value)
RUN_ORDER = [(seed, beta) for seed in SEEDS for beta in BETA_VALUES]   # 5 back-to-back pairs = 10 runs
SPLIT_SEED = 42                          # mission-level train/val/test split (independent of the training seed)

# --- Dataset / split ---
DATASET_REPO_URL = "https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git"
WORK_DIR = Path("/kaggle/working")
DATASET_BASE = WORK_DIR / "tii_uav_dataset"
DATASET_ROOT = DATASET_BASE / "Dataset"
TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15
N_CLASSES = 5                            # propeller-fault severity 0..4

# --- Windowing ---
WINDOW_SEC = 2.56
STEP_SEC = 1.28
MIN_SENSOR_SAMPLES_PER_WINDOW = 10

# --- Audio: strongest-64 bins (train-only), sqrt(|rFFT|), no window, no normalization ---
N_AUDIO_BINS = 64                        # frequency axis
TA_AUDIO = 64                            # audio time positions per window

# --- Sensor: raw 6 channels (accel x/y/z, gyro x/y/z), no derived features ---
N_SENSOR_CHANNELS = 6
TS_SENSOR = 128                          # sensor time positions per window
SENSOR_STD_EPS = 1e-6

# --- Model ---
D_MODEL = 256
TIME_EMBED_FREQUENCIES = 64
LACA_NUM_HEADS = 4
LACA_BETA = 1.0                          # finalized default; each run passes its own beta explicitly
LAG_BIAS_HIDDEN = 32                     # mlp g(Δτ): Linear(1,32) -> Tanh -> Linear(32,1) = 97 params
LAG_BIAS_PARAMS = 97                     # the LayerNorm added in v18 has no parameters -> still 97
LAG_BIAS_LN_EPS = 1e-5                   # LayerNorm over each window's (T_m, T_n) lag-bias grid, no affine
BETA_WARMUP_EPOCHS = 5                   # beta: linear 0 -> target over epochs 1..5 (0.2, 0.4, ..., 1.0), then held
MAMBA_NUM_BLOCKS = 2
MAMBA_D_STATE = 16
MAMBA_D_CONV = 2
MAMBA_EXPAND = 2
EXPECTED_MAMBA_PARAMS = 874_496          # 2 x 437,248 (real mamba_ssm)
DROPOUT = 0.15

# --- Loss: L_cls (class-weighted CE) + 0.1 * L_align (InfoNCE) + 0.1 * L_cons (KL) ---
LAMBDA_ALIGN = 0.1
LAMBDA_CONS = 0.1
INFONCE_TEMPERATURE = 0.07

# --- Optimization ---
EPOCHS = 20
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
LR_MIN = 1e-6
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 2

# --- Outputs ---
CHECKPOINT_DIR = WORK_DIR / "v18_checkpoints"
PROBS_DIR = WORK_DIR / "v18_test_probs"
RESULTS_CSV = WORK_DIR / "beta_stabilized_paired_results.csv"
for _d in (CHECKPOINT_DIR, PROBS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# --- Reference numbers ---
# v16 baseline (un-stabilized architecture). v18 changes the architecture, so it is NOT expected to reproduce this.
V16_BASELINE_SEED42 = {"accuracy": 0.9080, "macro_f1": 0.9048, "balanced_accuracy": 0.9022, "macro_auroc": 0.9894}
SAME_SEED_TOLERANCE = 0.0005             # same-seed reproducibility (code correctness), not a finding threshold
# v17 (un-stabilized) paired result, as reported: macro-F1 beta=1 minus beta=0 over seeds 42/123/2024/7/99.
V17_PAIRED = {"mean_diff": 0.0008, "sd_diff": 0.0020, "wins": 3, "n": 5, "lag_min": 0.23, "lag_max": 0.68}
# Optional: v17's per-run CSV (attach it as a Kaggle input, or leave it in /kaggle/working). If found, the
# beta=0 arm is checked for EXACT reproduction of v17, and v17's paired numbers are recomputed from it.
V17_RESULTS_CSV_CANDIDATES = ["/kaggle/input/**/beta_paired_replication_results.csv",
                              str(WORK_DIR / "beta_paired_replication_results.csv")]
SEED_TO_SEED_STD_F1 = 0.0160             # v15/v16 measured seed-to-seed std of test macro-F1 (beta=1.0)
PAIRED_ALPHA = 0.05
REFERENCE_SEED42_RNG_PREFIX = "5641b401720d"   # torch CPU RNG right after seed-42 model construction (v10-v15)
REFERENCE_DATA = {
    "bin_indices": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255,
                    258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 501, 502, 503, 504, 505, 506, 507, 508,
                    509, 510, 511, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 754, 757, 758, 759, 760,
                    761, 762, 763, 764, 765, 766],
    "sensor_fingerprint": {
        "train": (4934, -6167105.225, 808427.456, (906, 1020, 1004, 1008, 996)),
        "val":   (954, -1193206.652, 156301.664, (126, 229, 199, 190, 210)),
        "test":  (1130, -1413563.944, 185127.438, (286, 205, 210, 228, 201)),
    },
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")


def set_all_seeds(seed: int) -> None:
    """Seed python `random`, numpy, torch CPU and every CUDA device; deterministic cuDNN."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


print(f"STABILIZED paired beta replication: beta {BETA_VALUES} x seeds {SEEDS} = {len(RUN_ORDER)} runs | results -> {RESULTS_CSV}")
print(f"Lag-bias stabilization: LayerNorm over each window's (T_m, T_n) grid (eps {LAG_BIAS_LN_EPS}, no affine) | "
      f"zero-initialized g final layer | beta warm-up over {BETA_WARMUP_EPOCHS} epochs")
print(f"Fixed configuration: audio strongest-{N_AUDIO_BINS}, Ta={TA_AUDIO}, sqrt, no window, no normalization "
      f"| sensor raw {N_SENSOR_CHANNELS} ch, Ts={TS_SENSOR} | LACA beta=<per run>, {LACA_NUM_HEADS} heads, mlp g(Δτ) "
      f"{LAG_BIAS_PARAMS} params | Mamba {MAMBA_NUM_BLOCKS} x (d_state {MAMBA_D_STATE}, d_conv {MAMBA_D_CONV}) | "
      f"loss CE + {LAMBDA_ALIGN} InfoNCE + {LAMBDA_CONS} KL | {EPOCHS} epochs, batch {BATCH_SIZE}, AdamW {LEARNING_RATE}")

Device: cuda (Tesla T4)
STABILIZED paired beta replication: beta [0.0, 1.0] x seeds [42, 123, 2024, 7, 99] = 10 runs | results -> /kaggle/working/beta_stabilized_paired_results.csv
Lag-bias stabilization: LayerNorm over each window's (T_m, T_n) grid (eps 1e-05, no affine) | zero-initialized g final layer | beta warm-up over 5 epochs
Fixed configuration: audio strongest-64, Ta=64, sqrt, no window, no normalization | sensor raw 6 ch, Ts=128 | LACA beta=<per run>, 4 heads, mlp g(Δτ) 97 params | Mamba 2 x (d_state 16, d_conv 2) | loss CE + 0.1 InfoNCE + 0.1 KL | 20 epochs, batch 32, AdamW 0.0001


In [2]:
# ==============================================================================
# mamba_ssm installation + fusion-backbone resolution.
# Order: an already-installed package, else a prebuilt wheel matched to this
# Python / torch / CUDA / C++ ABI (the reliable path on Kaggle), else a
# time-capped source build. If none succeeds, or there is no CUDA GPU, the
# pure-PyTorch fallback block is used and announced below.
# ==============================================================================
PIP_TIMEOUT_S = 240


def _env_tags():
    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    torch_mm = ".".join(torch.__version__.split("+")[0].split(".")[:2])
    try:
        cxx11_abi = bool(torch._C._GLIBCXX_USE_CXX11_ABI)
    except AttributeError:
        cxx11_abi = False
    return py_tag, torch_mm, torch.version.cuda, cxx11_abi


def _prebuilt_wheel_url(repo, py_tag, torch_mm, cuda_ver, cxx11_abi):
    """Best-matching .whl asset in `repo`'s GitHub releases (exact CUDA version first, then CUDA major)."""
    req = urllib.request.Request(f"https://api.github.com/repos/{repo}/releases",
                                 headers={"User-Agent": "kaggle-notebook", "Accept": "application/vnd.github+json"})
    try:
        with urllib.request.urlopen(req, timeout=20) as resp:
            releases = json.load(resp)
    except Exception as exc:
        print(f"    could not query github.com/{repo}/releases: {exc!r}")
        return None
    abi = f"cxx11abi{'TRUE' if cxx11_abi else 'FALSE'}"
    cuda_exact = f"cu{cuda_ver.replace('.', '')}"
    cuda_major = f"cu{cuda_ver.split('.')[0]}"
    exact, major_only = [], []
    for rel in releases:
        for asset in rel.get("assets", []):
            name = asset.get("name", "")
            if not (name.endswith(".whl") and py_tag in name and abi in name and f"torch{torch_mm}" in name):
                continue
            if cuda_exact in name:
                exact.append(asset["browser_download_url"])
            elif cuda_major in name:
                major_only.append(asset["browser_download_url"])
    chosen = exact or major_only
    return chosen[0] if chosen else None


def _pip_install(args):
    """pip with a hard timeout and captured output, so a stuck build can never hang the kernel."""
    try:
        proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + args,
                              timeout=PIP_TIMEOUT_S, capture_output=True, text=True, check=False)
    except subprocess.TimeoutExpired:
        print(f"    pip install {args} exceeded {PIP_TIMEOUT_S}s — killed")
        return False
    if proc.returncode != 0:
        print(f"    pip install {args} failed (exit {proc.returncode}): {(proc.stderr or proc.stdout)[-400:]}")
    return proc.returncode == 0


def install_mamba_ssm():
    if importlib.util.find_spec("mamba_ssm") is not None:
        return "already installed"
    tags = _env_tags()
    print(f"Installing mamba_ssm + causal-conv1d (python={tags[0]}, torch={tags[1]}, cuda={tags[2]}, cxx11abi={tags[3]})")
    urls = [_prebuilt_wheel_url(repo, *tags) for repo in ("Dao-AILab/causal-conv1d", "state-spaces/mamba")]
    if all(urls) and all(_pip_install([u, "--no-deps"]) for u in urls):
        return "prebuilt wheel"
    print("  no complete prebuilt-wheel match — trying a time-capped source build")
    if _pip_install(["causal-conv1d>=1.2.0", "--no-build-isolation"]) and _pip_install(["mamba-ssm", "--no-build-isolation"]):
        return "source build"
    return "failed"


MAMBA_CLASS = None
if torch.cuda.is_available():
    print(f"mamba_ssm install: {install_mamba_ssm()}")
    try:
        from mamba_ssm import Mamba as MAMBA_CLASS
    except ImportError as exc:
        print(f"mamba_ssm import failed: {exc!r}")

if MAMBA_CLASS is not None:
    FUSION_BACKBONE = "mamba_ssm.Mamba (real selective-scan SSM, CUDA)"
    print(f"✅ FUSION BACKBONE: {FUSION_BACKBONE}")
else:
    FUSION_BACKBONE = "FALLBACK pure-PyTorch gated-conv block (NOT a state-space model)"
    print("🚨" * 45)
    print(f"🚨  FUSION BACKBONE: {FUSION_BACKBONE}")
    print("🚨  " + ("no CUDA GPU detected" if not torch.cuda.is_available() else "mamba_ssm could not be installed/imported")
          + " — results will NOT match the reference numbers, which were produced with real mamba_ssm.")
    print("🚨" * 45)

Installing mamba_ssm + causal-conv1d (python=cp312, torch=2.10, cuda=12.8, cxx11abi=True)
    pip install ['https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.7.0/causal_conv1d-1.7.0%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_aarch64.whl', '--no-deps'] failed (exit 1): ERROR: causal_conv1d-1.7.0+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_aarch64.whl is not a supported wheel on this platform.

  no complete prebuilt-wheel match — trying a time-capped source build
mamba_ssm install: source build
✅ FUSION BACKBONE: mamba_ssm.Mamba (real selective-scan SSM, CUDA)


## 3. Data loading & preprocessing
Mission-level split, strongest-64 bin selection from training audio only, 2.56 s windows with real timestamps, and sensor standardization with training-split statistics. Leakage and timestamp checks are included.

In [3]:
# ==============================================================================
# DATA — TII UAV Realistic Fault Dataset: parse missions -> mission-level split
# -> strongest-64 bins from TRAINING audio only (frozen) -> 2.56 s windows with
# real per-window timestamps -> datasets (sensor standardized with TRAINING stats).
# ==============================================================================
if not DATASET_ROOT.exists():
    print(f"Cloning {DATASET_REPO_URL} ...")
    subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO_URL, str(DATASET_BASE)], check=True)


def parse_mission(mission_dir: Path):
    """Real 6-axis IMU (seconds from mission start) + raw audio chunks with per-chunk timestamps.
    Audio chunk duration = the mission's IMU-measured duration / number of chunks (both come from the same
    continuous recording; the audio log carries no per-chunk timestamp)."""
    sensor_files = list(mission_dir.glob("*SensorCombined*.jsonl"))
    audio_files = list(mission_dir.glob("*AudioBuffer*.jsonl"))
    if not sensor_files or not audio_files:
        return None

    sensor_times, accel, gyro = [], [], []
    with open(sensor_files[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
                t_sec = float(rec["timestamp"]) / 1e6
                sensor_times.append(t_sec)
                accel.append(rec.get("accelerometer_m_s2", [0.0, 0.0, 9.81]))
                gyro.append(rec.get("gyro_rad", [0.0, 0.0, 0.0]))
            except Exception:
                continue
    if len(sensor_times) < 50:
        return None
    sensor_times = np.array(sensor_times, dtype=np.float64)
    sensor_times = sensor_times - sensor_times[0]
    telemetry = np.concatenate([np.array(accel, dtype=np.float32), np.array(gyro, dtype=np.float32)], axis=-1)

    audio_chunks = []
    with open(audio_files[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
                chunk = rec.get("audio_data", rec.get("buffer", rec.get("data", [])))
                if len(chunk) > 0:
                    audio_chunks.append(chunk)
            except Exception:
                continue
    if len(audio_chunks) < 10:
        return None
    audio_chunks = np.array(audio_chunks, dtype=np.float32)

    chunk_duration_s = (sensor_times[-1] - sensor_times[0]) / len(audio_chunks)
    audio_times = np.arange(len(audio_chunks), dtype=np.float64) * chunk_duration_s
    return {"sensor_times": sensor_times, "telemetry": telemetry, "audio_times": audio_times, "audio_chunks": audio_chunks}


def split_missions(dataset_root: Path):
    """Mission-level stratified split (70 / 15 / 15 per class) with its own fixed seed."""
    class_missions = defaultdict(list)
    for class_dir in sorted(dataset_root.iterdir()):
        if not class_dir.is_dir() or class_dir.name.startswith("."):
            continue
        try:
            label = int(class_dir.name)
        except ValueError:
            continue
        for mission_dir in sorted(class_dir.iterdir()):
            if mission_dir.is_dir() and not mission_dir.name.startswith("."):
                class_missions[label].append(mission_dir)

    random.seed(SPLIT_SEED)
    splits = {"train": {}, "val": {}, "test": {}}
    for label in sorted(class_missions):
        missions = class_missions[label]
        random.shuffle(missions)
        n_train = max(1, int(len(missions) * TRAIN_FRACTION))
        n_val = max(1, int(len(missions) * VAL_FRACTION))
        splits["train"][label] = missions[:n_train]
        splits["val"][label] = missions[n_train:n_train + n_val]
        splits["test"][label] = missions[n_train + n_val:]
        print(f"Class {label} ({len(missions)} flights) -> train {n_train}, val {len(splits['val'][label])}, "
              f"test {len(splits['test'][label])}")
    return splits


def select_strongest_bins(train_missions, n_bins):
    """Indices of the n_bins rFFT magnitude bins with the highest mean magnitude over TRAINING audio only
    (unsupervised energy, no labels). Returned sorted; frozen and reused for val/test."""
    magnitude_sum, n_chunks, fft_len = None, 0, None
    for mission in train_missions:
        mag = np.abs(np.fft.rfft(mission["audio_chunks"], axis=1))
        fft_len = mag.shape[1] if fft_len is None else min(fft_len, mag.shape[1])
        mag = mag[:, :fft_len]
        if magnitude_sum is not None and len(magnitude_sum) > fft_len:
            magnitude_sum = magnitude_sum[:fft_len]
        magnitude_sum = mag.sum(axis=0) if magnitude_sum is None else magnitude_sum + mag.sum(axis=0)
        n_chunks += mag.shape[0]
    mean_magnitude = magnitude_sum / n_chunks
    return np.sort(np.argsort(mean_magnitude)[::-1][:n_bins]).astype(int)


def audio_features(audio_chunks, bin_indices):
    """raw chunk -> rFFT (no window) -> |.| -> frozen strongest bins -> sqrt (no normalization)."""
    mag = np.abs(np.fft.rfft(audio_chunks, axis=1))
    valid = bin_indices[bin_indices < mag.shape[1]]
    feats = np.zeros((mag.shape[0], len(bin_indices)), dtype=np.float32)
    feats[:, :len(valid)] = np.sqrt(mag[:, valid]).astype(np.float32)
    return feats


def slice_windows(mission, label, bin_indices):
    """2.56 s windows (1.28 s step). Sensor: TS_SENSOR real IMU samples chosen evenly by index inside the window;
    audio: TA_AUDIO real chunk timestamps chosen the same way (a window with fewer than 2 audio chunks — the
    recording edge — falls back to an even grid). tau_a / tau_s = those REAL timestamps relative to window start."""
    s_times, telemetry = mission["sensor_times"], mission["telemetry"]
    a_times = mission["audio_times"]
    a_feats = audio_features(mission["audio_chunks"], bin_indices)
    max_time = min(s_times[-1], a_times[-1])

    windows, t_start = [], 0.0
    while t_start + WINDOW_SEC <= max_time:
        t_end = t_start + WINDOW_SEC
        s_mask = (s_times >= t_start) & (s_times <= t_end)
        if s_mask.sum() < MIN_SENSOR_SAMPLES_PER_WINDOW:
            t_start += STEP_SEC
            continue

        s_idx = np.where(s_mask)[0]
        tau_s_abs = s_times[s_idx[np.linspace(0, len(s_idx) - 1, TS_SENSOR).round().astype(int)]]
        x_s = np.zeros((TS_SENSOR, N_SENSOR_CHANNELS), dtype=np.float32)
        for c in range(N_SENSOR_CHANNELS):
            x_s[:, c] = np.interp(tau_s_abs, s_times[s_mask], telemetry[s_mask, c])

        a_mask = (a_times >= t_start) & (a_times <= t_end)
        if a_mask.sum() >= 2:
            a_idx = np.where(a_mask)[0]
            tau_a_abs = a_times[a_idx[np.linspace(0, len(a_idx) - 1, TA_AUDIO).round().astype(int)]]
        else:
            tau_a_abs = np.linspace(t_start, t_end, TA_AUDIO)
        x_a = np.zeros((TA_AUDIO, len(bin_indices)), dtype=np.float32)
        for c in range(len(bin_indices)):
            x_a[:, c] = np.interp(tau_a_abs, a_times, a_feats[:, c], left=0.0, right=0.0)

        windows.append({
            "x_a": x_a,
            "x_s": x_s,
            "label": int(label),
            "tau_a": (tau_a_abs - t_start).astype(np.float32),
            "tau_s": (tau_s_abs - t_start).astype(np.float32),
            "window_sec": np.float32(WINDOW_SEC),
        })
        t_start += STEP_SEC
    return windows


class UAVWindowDataset(Dataset):
    """Windows -> tensors. Sensor channels are standardized with mean/std that must come from the TRAINING
    split (computed here only when none are passed, i.e. for the training set). tau is normalized by the
    window length."""

    def __init__(self, windows, sensor_mean=None, sensor_std=None):
        self.windows = windows
        if sensor_mean is None or sensor_std is None:
            all_s = np.concatenate([w["x_s"] for w in windows], axis=0)
            sensor_mean = all_s.mean(axis=0, keepdims=True)
            sensor_std = all_s.std(axis=0, keepdims=True) + SENSOR_STD_EPS
        self.sensor_mean, self.sensor_std = sensor_mean, sensor_std

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        w = self.windows[idx]
        window_sec = float(w["window_sec"])
        return {
            "x_a": torch.tensor(w["x_a"], dtype=torch.float32),
            "x_s": torch.tensor((w["x_s"] - self.sensor_mean) / self.sensor_std, dtype=torch.float32),
            "tau_a": torch.tensor(w["tau_a"], dtype=torch.float32) / window_sec,
            "tau_s": torch.tensor(w["tau_s"], dtype=torch.float32) / window_sec,
            "label": torch.tensor(w["label"], dtype=torch.long),
        }


def split_fingerprint(windows):
    """(n windows, sum of raw sensor values, sum of sensor timestamps, class counts) — deterministic data check."""
    x_s = np.stack([w["x_s"] for w in windows]).astype(np.float64)
    tau_s = np.stack([w["tau_s"] for w in windows]).astype(np.float64)
    counts = tuple(np.bincount([w["label"] for w in windows], minlength=N_CLASSES).tolist())
    return (len(windows), round(float(x_s.sum()), 3), round(float(tau_s.sum()), 3), counts)


# --- 1. split + parse ---
mission_splits = split_missions(DATASET_ROOT)
split_dirs = {s: [m for label in sorted(mission_splits[s]) for m in mission_splits[s][label]] for s in mission_splits}
assert not set(split_dirs["train"]) & (set(split_dirs["val"]) | set(split_dirs["test"])), "LEAKAGE: mission in two splits"
parsed_missions = {m: parse_mission(m) for s in split_dirs for m in tqdm(split_dirs[s], desc=f"parse {s}", leave=False)}

# --- 2. strongest-64 bins from TRAINING audio only ---
bin_source_missions = [m for m in split_dirs["train"] if parsed_missions[m] is not None]
assert set(bin_source_missions) <= set(split_dirs["train"])
SELECTED_BINS = select_strongest_bins([parsed_missions[m] for m in bin_source_missions], N_AUDIO_BINS)
assert len(SELECTED_BINS) == N_AUDIO_BINS
print(f"[LEAKAGE CHECK] strongest-{N_AUDIO_BINS} bins computed from {len(bin_source_missions)} TRAINING missions only "
      f"(0 of {len(split_dirs['val']) + len(split_dirs['test'])} val/test missions) -> frozen for all splits: "
      f"{SELECTED_BINS.tolist()}")

# --- 3. windows (the same frozen bins for every split) ---
split_windows = {}
for split in ("train", "val", "test"):
    split_windows[split] = []
    for label in sorted(mission_splits[split]):
        for m in mission_splits[split][label]:
            if parsed_missions[m] is not None:
                split_windows[split].extend(slice_windows(parsed_missions[m], label, SELECTED_BINS))
    print(f"{split:<5}: {len(split_windows[split])} windows | audio {TA_AUDIO}x{N_AUDIO_BINS}, sensor {TS_SENSOR}x{N_SENSOR_CHANNELS}")
DATA_FINGERPRINT = {s: split_fingerprint(split_windows[s]) for s in split_windows}

# --- 4. datasets: sensor mean/std from the TRAINING split only, reused for val/test ---
train_ds = UAVWindowDataset(split_windows["train"])
val_ds = UAVWindowDataset(split_windows["val"], train_ds.sensor_mean, train_ds.sensor_std)
test_ds = UAVWindowDataset(split_windows["test"], train_ds.sensor_mean, train_ds.sensor_std)
_train_s = np.concatenate([w["x_s"] for w in split_windows["train"]], axis=0)
_val_s = np.concatenate([w["x_s"] for w in split_windows["val"]], axis=0)
assert val_ds.sensor_mean is train_ds.sensor_mean and test_ds.sensor_std is train_ds.sensor_std
assert np.array_equal(train_ds.sensor_mean, _train_s.mean(axis=0, keepdims=True))
print(f"[LEAKAGE CHECK] sensor standardization stats = TRAINING-split stats (same objects reused for val/test) | "
      f"train mean {np.round(train_ds.sensor_mean.ravel(), 3).tolist()} vs val's own mean "
      f"{np.round(_val_s.mean(axis=0), 3).tolist()} (not used)")
del _train_s, _val_s

# --- 5. real timestamps reach the model: tau must differ across windows ---
_tau_a = np.stack([train_ds[i]["tau_a"].numpy() for i in range(len(train_ds))])
_tau_s = np.stack([train_ds[i]["tau_s"].numpy() for i in range(len(train_ds))])
_n_unique_a, _n_unique_s = len(np.unique(_tau_a, axis=0)), len(np.unique(_tau_s, axis=0))
assert _n_unique_a > 1 and _n_unique_s > 1, "real timestamps are NOT reaching the model (tau identical for every window)"
assert not np.allclose(_tau_s, np.linspace(0.0, 1.0, TS_SENSOR)[None, :]), "tau_s equals a synthetic linspace grid"
print(f"[TIMESTAMP CHECK] {len(train_ds)} training windows: {_n_unique_a} distinct tau_a rows, {_n_unique_s} distinct "
      f"tau_s rows -> real per-window timestamps reach HTT/LACA ✅")
del _tau_a, _tau_s

Cloning https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git ...


Cloning into '/kaggle/working/tii_uav_dataset'...


Class 0 (19 flights) -> train 13, val 2, test 4
Class 1 (20 flights) -> train 14, val 3, test 3
Class 2 (20 flights) -> train 14, val 3, test 3
Class 3 (20 flights) -> train 14, val 3, test 3
Class 4 (20 flights) -> train 14, val 3, test 3


Updating files: 100% (217/217), done.


parse train:   0%|          | 0/69 [00:00<?, ?it/s]

parse val:   0%|          | 0/14 [00:00<?, ?it/s]

parse test:   0%|          | 0/16 [00:00<?, ?it/s]

[LEAKAGE CHECK] strongest-64 bins computed from 69 TRAINING missions only (0 of 30 val/test missions) -> frozen for all splits: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 754, 757, 758, 759, 760, 761, 762, 763, 764, 765, 766]
train: 4934 windows | audio 64x64, sensor 128x6
val  : 954 windows | audio 64x64, sensor 128x6
test : 1130 windows | audio 64x64, sensor 128x6
[LEAKAGE CHECK] sensor standardization stats = TRAINING-split stats (same objects reused for val/test) | train mean [0.05299999937415123, -0.0430000014603138, -9.78600025177002, 0.004000000189989805, 0.003000000026077032, -0.0] vs val's own mean [0.029999999329447746, -0.061000000685453415, -9.748000144958496, 0.004999999888241291, 0.003000000026077032, -0.0] (not used)
[TIMESTAMP CHECK] 4934 training windows: 4934 distinct 

## 4. Model definition
Encoders → HTT → bi-directional LACA → 2 Mamba blocks → attention pooling → classifier. In v18 the LACA lag-bias term is **β · LN_grid(g(Δτ))**:
- g's final layer is zero-initialized;
- the LayerNorm runs over each window's (T_m, T_n) grid and has no parameters;
- `model.set_beta()` sets the β in effect, which the warm-up schedule changes every epoch.

This cell prints the exact LayerNorm placement and tensor shapes from a real batch.

In [4]:
# ==============================================================================
# MODEL — encoders -> HTT (modality + continuous-time embedding of the REAL tau)
# -> bi-directional LACA (lag-aware cross-attention, STABILIZED lag bias: beta * LN_grid(g(Δτ)),
#    zero-init g final layer, beta warm-up; 4 heads)
# -> [H_a ; SEP ; H_s] -> 2 Mamba blocks -> attention pooling -> classifier.
# Auxiliary per-modality heads feed only the consistency loss.
# ==============================================================================
class ContinuousTimeEmbedding(nn.Module):
    def __init__(self, d_model: int, num_frequencies: int = TIME_EMBED_FREQUENCIES):
        super().__init__()
        self.frequencies = nn.Parameter(
            torch.exp(torch.linspace(0, math.log(1000.0), num_frequencies)), requires_grad=False
        )
        self.mlp = nn.Sequential(
            nn.Linear(2 * num_frequencies, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

    def forward(self, tau: torch.Tensor) -> torch.Tensor:
        if tau.dim() == 2:
            tau = tau.unsqueeze(-1)
        angles = tau * self.frequencies * 2.0 * math.pi
        return self.mlp(torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1))


class HeterogeneousTemporalTokenization(nn.Module):
    """token = LayerNorm(features + modality embedding + time embedding(tau)); modality 0 = audio, 1 = sensor."""

    def __init__(self, d_model: int):
        super().__init__()
        self.d_model = d_model
        self.modality_embeddings = nn.Embedding(2, d_model)
        self.time_embedding = ContinuousTimeEmbedding(d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, z: torch.Tensor, tau: torch.Tensor, modality_idx: int) -> torch.Tensor:
        mod_idx = torch.tensor(modality_idx, device=z.device, dtype=torch.long)
        e_m = self.modality_embeddings(mod_idx).view(1, 1, self.d_model)
        return self.norm(z + e_m + self.time_embedding(tau))


class GridLayerNorm(nn.Module):
    """LayerNorm for the lag-bias grid (v18).

    g(Δτ) produces ONE scalar per (query i, key j) pair, so its output is a (B, T_m, T_n) grid with no feature
    dimension > 1. A LayerNorm over a size-1 dimension would subtract each value from itself and return 0 for every
    input. Instead, the normalization is taken over each window's whole (T_m, T_n) grid, i.e. every bias value that
    enters that window's attention matrix. It is computed separately for each window: every window has its own real
    timestamps, and mixing windows would make one window's bias depend on the rest of the batch and differ between
    training and evaluation. There are no affine parameters, so beta is the only source of scale.

    Consequences: the subtracted mean is one constant per window, and softmax is unchanged by a constant added to
    every score, so only the division by the std affects attention. For an all-zero grid (the zero-init state) the
    output is exactly 0. As g develops variance v, the output std grows smoothly as sqrt(v / (v + eps)) toward 1."""

    def __init__(self, eps: float = LAG_BIAS_LN_EPS):
        super().__init__()
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:   # x: (B, T_m, T_n)
        return F.layer_norm(x, x.shape[-2:], eps=self.eps)


class PairwiseLACA(nn.Module):
    """Cross-attention from modality m to n with a learned lag bias g(Δτ) added to the scores (scaled by β)."""

    def __init__(self, d_model: int, num_heads: int = LACA_NUM_HEADS, beta: float = LACA_BETA):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.beta = beta
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.lag_bias = nn.Sequential(nn.Linear(1, LAG_BIAS_HIDDEN), nn.Tanh(), nn.Linear(LAG_BIAS_HIDDEN, 1))
        # v18 ZERO-INIT: g's final Linear starts at exactly 0 -> g(Δτ) = 0 for every Δτ and every seed. The layer was
        # already built (same RNG draws as v16), and nn.init.zeros_ draws no random numbers, so every other
        # layer's initial weights and the training RNG stream are unchanged.
        nn.init.zeros_(self.lag_bias[-1].weight)
        nn.init.zeros_(self.lag_bias[-1].bias)
        self.lag_bias_norm = GridLayerNorm()   # v18: no parameters

    def lag_bias_term(self, tau_m, tau_n):
        """LN_grid(g(Δτ)), BEFORE beta. Returns (B, 1, T_m, T_n), broadcast over the heads."""
        delta_tau = (tau_m.unsqueeze(2) - tau_n.unsqueeze(1)).unsqueeze(-1)   # (B, T_m, T_n, 1)
        g = self.lag_bias(delta_tau).squeeze(-1)                                # (B, T_m, T_n): one scalar per (i, j)
        return self.lag_bias_norm(g).unsqueeze(1)                               # normalized per window -> (B, 1, T_m, T_n)

    def forward(self, H_m, H_n, tau_m, tau_n):
        B, T_m, _ = H_m.shape
        _, T_n, _ = H_n.shape
        Q = self.w_q(H_m).view(B, T_m, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.w_k(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.w_v(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        lag_bias = self.lag_bias_term(tau_m, tau_n)

        attn = F.softmax(scores + self.beta * lag_bias, dim=-1)   # beta = the value in effect (warm-up schedule)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, T_m, self.d_model)

        attn_mean = attn.mean(dim=1)
        tau_diff = tau_n.unsqueeze(1) - tau_m.unsqueeze(2)
        expected_lag = (attn_mean * tau_diff).sum(dim=(1, 2)) / float(T_m)
        return self.out_proj(out), expected_lag


class BiModalLACAFusion(nn.Module):
    def __init__(self, d_model: int, beta: float = LACA_BETA):
        super().__init__()
        self.laca_as = PairwiseLACA(d_model, beta=beta)   # beta scales the lag bias in BOTH directions
        self.laca_sa = PairwiseLACA(d_model, beta=beta)
        self.gamma_as = nn.Parameter(torch.tensor(0.5))
        self.gamma_sa = nn.Parameter(torch.tensor(0.5))

    def forward(self, Ha, Hs, tau_a, tau_s):
        H_a_from_s, lag_as = self.laca_as(Ha, Hs, tau_a, tau_s)
        H_s_from_a, lag_sa = self.laca_sa(Hs, Ha, tau_s, tau_a)
        return Ha + self.gamma_as * H_a_from_s, Hs + self.gamma_sa * H_s_from_a, {"lag_as": lag_as, "lag_sa": lag_sa}


class MambaBlock(nn.Module):
    """Pre-norm residual wrapper around the real mamba_ssm.Mamba selective-scan SSM."""

    def __init__(self, d_model: int, d_state: int, d_conv: int, expand: int):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mamba = MAMBA_CLASS(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.mamba(self.norm(x))


class FallbackGatedConvBlock(nn.Module):
    """CPU-only stand-in when mamba_ssm is unavailable: depthwise causal conv + gated MLP. NOT a state-space
    model (no hidden state, no A/B/C, no discretization)."""

    def __init__(self, d_model: int, d_conv: int, expand: int):
        super().__init__()
        d_inner = expand * d_model
        self.in_proj = nn.Linear(d_model, d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(d_inner, d_inner, kernel_size=d_conv, padding=d_conv - 1, groups=d_inner, bias=True)
        self.mixer = nn.Sequential(nn.Linear(d_inner, d_inner), nn.GELU(), nn.Linear(d_inner, d_inner))
        self.out_proj = nn.Linear(d_inner, d_model, bias=False)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        L = x.shape[1]
        x_branch, z_branch = self.in_proj(self.norm(x)).chunk(2, dim=-1)
        x_conv = F.silu(self.conv1d(x_branch.transpose(1, 2))[:, :, :L].transpose(1, 2))
        return x + self.out_proj(self.mixer(x_conv) * F.silu(z_branch))


def build_fusion_blocks(d_model, num_blocks, d_state, d_conv, expand=MAMBA_EXPAND):
    if MAMBA_CLASS is not None:
        return nn.ModuleList([MambaBlock(d_model, d_state, d_conv, expand) for _ in range(num_blocks)])
    return nn.ModuleList([FallbackGatedConvBlock(d_model, d_conv, expand) for _ in range(num_blocks)])


# The reference results were produced by a model whose constructor drew, from the torch RNG, the initial weights
# of a 4-block (d_state 16, d_conv 4) Mamba stack before the final 2-block stack was built from the SAME starting
# RNG state. Every module built afterwards (pooling, classifier, auxiliary heads) and the whole training RNG stream
# (shuffle order, dropout masks) depend on the RNG state that 4-block draw leaves behind. To reproduce those
# results exactly, the constructor performs the same draw and discards it; the discarded blocks are never part of
# the model. This is a fixed step of weight initialization, not a configuration option.
RNG_REFERENCE_STACK = (4, 16, 4)   # (num_blocks, d_state, d_conv) whose initialization is drawn and discarded


class AeroSyncMamba(nn.Module):
    def __init__(self, d_model: int = D_MODEL, n_classes: int = N_CLASSES, beta: float = LACA_BETA):
        super().__init__()
        self.d_model = d_model
        self.n_classes = n_classes
        self.beta_target = float(beta)   # target of the warm-up schedule

        self.sensor_encoder = nn.Sequential(
            nn.Conv1d(N_SENSOR_CHANNELS, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, d_model, kernel_size=3, padding=1), nn.BatchNorm1d(d_model), nn.GELU(),
        )
        self.audio_encoder = nn.Sequential(
            nn.Linear(N_AUDIO_BINS, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Linear(d_model, d_model)
        )
        self.htt = HeterogeneousTemporalTokenization(d_model)
        self.laca = BiModalLACAFusion(d_model, beta=self.beta_target)   # plain float: no params, no RNG draws
        self.sep_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        rng_before = torch.get_rng_state()
        build_fusion_blocks(d_model, *RNG_REFERENCE_STACK)
        rng_after_reference = torch.get_rng_state()
        torch.set_rng_state(rng_before)
        self.mamba_blocks = build_fusion_blocks(d_model, MAMBA_NUM_BLOCKS, MAMBA_D_STATE, MAMBA_D_CONV)
        torch.set_rng_state(rng_after_reference)

        self.pool_attn = nn.Sequential(nn.Linear(d_model, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, n_classes),
        )
        self.aux_a = nn.Linear(d_model, n_classes)
        self.aux_s = nn.Linear(d_model, n_classes)

    def set_beta(self, beta: float) -> None:
        """The beta in effect for BOTH LACA directions (the warm-up schedule changes it every epoch)."""
        self.laca.laca_as.beta = float(beta)
        self.laca.laca_sa.beta = float(beta)

    def forward(self, xa, xs, ta, ts):
        B = xa.shape[0]
        za = self.audio_encoder(xa)
        zs = self.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)

        Ha = self.htt(za, ta, modality_idx=0)
        Hs = self.htt(zs, ts, modality_idx=1)
        H_tilde_a, H_tilde_s, lags = self.laca(Ha, Hs, ta, ts)
        za_p = H_tilde_a.mean(dim=1)
        zs_p = H_tilde_s.mean(dim=1)

        F_seq = torch.cat([H_tilde_a, self.sep_token.expand(B, -1, -1), H_tilde_s], dim=1)
        for block in self.mamba_blocks:
            F_seq = block(F_seq)

        weights = F.softmax(self.pool_attn(F_seq), dim=1)
        zf = (F_seq * weights).sum(dim=1)
        logits = self.classifier(zf)

        pooled = {"z_a": za_p, "z_s": zs_p}
        aux_logits = {"y_a": self.aux_a(za_p), "y_s": self.aux_s(zs_p)}
        return logits, pooled, aux_logits, lags


def beta_schedule(beta_target: float, epoch: int, warmup: int = BETA_WARMUP_EPOCHS) -> float:
    """beta in effect during `epoch` (1-indexed): linear 0 -> target over epochs 1..warmup, then held at target.
    For target 1.0: epoch 1 -> 0.2, 2 -> 0.4, ..., 5 -> 1.0, 6+ -> 1.0. For target 0.0: 0.0 at every epoch."""
    return float(beta_target) * min(epoch, warmup) / warmup


def verify_architecture(model, beta=LACA_BETA):
    """Finalized architecture, read from the constructed modules (beta = the value this model must use)."""
    for name in ("laca_as", "laca_sa"):
        laca = getattr(model.laca, name)
        assert laca.num_heads == LACA_NUM_HEADS, f"{name}: {laca.num_heads} heads"
        assert laca.beta == beta, f"{name}: beta {laca.beta} != requested {beta}"
        assert sum(p.numel() for p in laca.lag_bias.parameters()) == LAG_BIAS_PARAMS
        assert isinstance(laca.lag_bias_norm, GridLayerNorm) and not list(laca.lag_bias_norm.parameters())
    assert len(model.mamba_blocks) == MAMBA_NUM_BLOCKS
    mamba_params = sum(p.numel() for p in model.mamba_blocks.parameters())
    if MAMBA_CLASS is not None:
        for block in model.mamba_blocks:
            m = block.mamba
            assert (m.d_state, m.d_conv, m.expand) == (MAMBA_D_STATE, MAMBA_D_CONV, MAMBA_EXPAND)
        assert mamba_params == EXPECTED_MAMBA_PARAMS, f"Mamba params {mamba_params:,} != {EXPECTED_MAMBA_PARAMS:,}"
    return mamba_params


_probe_counts = {}
for _b in BETA_VALUES:
    _probe = AeroSyncMamba(beta=_b)
    verify_architecture(_probe, beta=_b)
    _probe_counts[_b] = sum(p.numel() for p in _probe.parameters())
    del _probe
assert len(set(_probe_counts.values())) == 1, f"parameter count depends on beta: {_probe_counts}"
_probe = AeroSyncMamba()
print(f"Model: {_probe_counts[BETA_VALUES[0]]:,} params for every beta in {BETA_VALUES} | fusion blocks: "
      f"{type(_probe.mamba_blocks[0]).__name__} x {len(_probe.mamba_blocks)} ({verify_architecture(_probe):,} params) | "
      f"LACA {LACA_NUM_HEADS} heads, g(Δτ) {LAG_BIAS_PARAMS} params | backbone: {FUSION_BACKBONE}")
del _probe


# --- LayerNorm placement: trace real tensor shapes through one LACA direction on 2 training windows ---
_ta = torch.stack([train_ds[i]["tau_a"] for i in range(2)])
_ts = torch.stack([train_ds[i]["tau_s"] for i in range(2)])
_probe = AeroSyncMamba()
with torch.no_grad():
    for _name, _laca, _tm, _tn in (("laca_as", _probe.laca.laca_as, _ta, _ts), ("laca_sa", _probe.laca.laca_sa, _ts, _ta)):
        _dt = (_tm.unsqueeze(2) - _tn.unsqueeze(1)).unsqueeze(-1)
        _g = _laca.lag_bias(_dt).squeeze(-1)
        _out = _laca.lag_bias_term(_tm, _tn)
        print(f"[LAYERNORM] {_name}: Δτ {tuple(_dt.shape)} -> g(Δτ) {tuple(_g.shape)} = (B, T_m, T_n) -> "
              f"LN over dims {tuple(_g.shape[-2:])} ({_g.shape[-2] * _g.shape[-1]:,} values per window, per window) -> "
              f"{tuple(_out.shape)} -> x beta -> added to scores (B, {LACA_NUM_HEADS} heads, T_m, T_n)")
        assert float(_g.abs().max()) == 0.0 and float(_out.abs().max()) == 0.0, "zero-init g should give an all-zero term"
del _probe, _ta, _ts
print("[LAYERNORM] placement: PairwiseLACA.forward -> scores + beta * LN_grid(g(Δτ)); the LayerNorm is applied to g's "
      "scalar output BEFORE beta.")
print("[LAYERNORM] why this shape: g outputs one scalar per (i, j) pair, so there is no feature dim > 1 (LN over a "
      "size-1 dim would output 0 for every input). Normalizing over each window's full (T_m, T_n) grid covers every "
      "bias value in that window's attention matrix. It is per window, never across the batch, so a window's bias "
      "does not depend on its batch-mates and is the same in train and eval. No affine: beta is the only scale. The "
      "subtracted mean is one constant per window, which softmax ignores; only the std normalization affects "
      "attention.")
print(f"[WARM-UP] beta schedule for target 1.0: " + ", ".join(f"ep{e}={beta_schedule(1.0, e):.1f}" for e in range(1, 8))
      + f", ... ep{EPOCHS}={beta_schedule(1.0, EPOCHS):.1f} | target 0.0: "
      + ("0.0 at every epoch" if all(beta_schedule(0.0, e) == 0.0 for e in range(1, EPOCHS + 1)) else "NOT ZERO (bug)"))

Model: 1,688,788 params for every beta in [0.0, 1.0] | fusion blocks: MambaBlock x 2 (874,496 params) | LACA 4 heads, g(Δτ) 97 params | backbone: mamba_ssm.Mamba (real selective-scan SSM, CUDA)
[LAYERNORM] laca_as: Δτ (2, 64, 128, 1) -> g(Δτ) (2, 64, 128) = (B, T_m, T_n) -> LN over dims (64, 128) (8,192 values per window, per window) -> (2, 1, 64, 128) -> x beta -> added to scores (B, 4 heads, T_m, T_n)
[LAYERNORM] laca_sa: Δτ (2, 128, 64, 1) -> g(Δτ) (2, 128, 64) = (B, T_m, T_n) -> LN over dims (128, 64) (8,192 values per window, per window) -> (2, 1, 128, 64) -> x beta -> added to scores (B, 4 heads, T_m, T_n)
[LAYERNORM] placement: PairwiseLACA.forward -> scores + beta * LN_grid(g(Δτ)); the LayerNorm is applied to g's scalar output BEFORE beta.
[LAYERNORM] why this shape: g outputs one scalar per (i, j) pair, so there is no feature dim > 1 (LN over a size-1 dim would output 0 for every input). Normalizing over each window's full (T_m, T_n) grid covers every bias value in that window

## 5. Training function
`train_one_run(seed, beta)` does the following:
- resets every RNG and builds a fresh model;
- **checks that g's final layer weight and bias are exactly 0 before any training step**;
- sets β from the warm-up schedule at the start of every epoch and reads it back from both LACA modules;
- measures max|β·LN(g(Δτ))| after **every** epoch. For β = 0.0 runs it asserts this is exactly 0 at every epoch.

It records the contribution at the final epoch and at the best checkpoint. The test set is evaluated with the best checkpoint and **the β it was validated with**.

In [6]:
# ==============================================================================
# TRAINING — loss, metrics, stabilization checks (zero-init, warm-up, per-epoch
# lag contribution), and train_one_run(seed, beta): one full model end-to-end
# (fresh model / optimizer / scheduler, beta warm-up, 20 epochs, best-validation
# checkpoint), evaluated on the test set.
# ==============================================================================
class AeroSyncLoss(nn.Module):
    """L = CE_classweighted(logits, y) + LAMBDA_ALIGN * InfoNCE(z_a, z_s) + LAMBDA_CONS * KL-consistency."""

    def __init__(self, class_weights):
        super().__init__()
        self.lambda_align = LAMBDA_ALIGN
        self.lambda_cons = LAMBDA_CONS
        self.temperature = INFONCE_TEMPERATURE
        self.ce = nn.CrossEntropyLoss(weight=class_weights)

    def _infonce(self, z1, z2):
        sim = torch.matmul(F.normalize(z1, p=2, dim=-1), F.normalize(z2, p=2, dim=-1).t()) / self.temperature
        labels = torch.arange(z1.shape[0], device=z1.device)
        return 0.5 * (F.cross_entropy(sim, labels) + F.cross_entropy(sim.t(), labels))

    def forward(self, logits, targets, pooled, aux_logits):
        loss_cls = self.ce(logits, targets)
        loss_align = self._infonce(pooled["z_a"], pooled["z_s"])
        p_fused = F.softmax(logits.detach(), dim=-1)
        loss_cons = (
            F.kl_div(F.log_softmax(aux_logits["y_a"], dim=-1), p_fused, reduction="batchmean") +
            F.kl_div(F.log_softmax(aux_logits["y_s"], dim=-1), p_fused, reduction="batchmean")
        ) / 2.0
        total = loss_cls + self.lambda_align * loss_align + self.lambda_cons * loss_cons
        return total, {"loss_cls": loss_cls.item(), "loss_align": loss_align.item(), "loss_cons": loss_cons.item()}


def compute_metrics(y_true, y_prob):
    """accuracy, macro-F1, balanced accuracy, macro one-vs-rest AUROC from class probabilities (prediction = argmax)."""
    y_pred = y_prob.argmax(axis=1)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_auroc": float(roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")),
    }


def train_class_weights(windows):
    """Inverse class frequency over the TRAINING windows."""
    counts = np.bincount([w["label"] for w in windows], minlength=N_CLASSES)
    return torch.tensor(counts.sum() / (len(counts) * np.maximum(counts, 1)), dtype=torch.float32)


def _batch_to_device(batch, non_blocking=False):
    return tuple(batch[k].to(device, non_blocking=non_blocking) for k in ("x_a", "x_s", "tau_a", "tau_s", "label"))


def predict_probs(model, loader):
    """Softmax probabilities and true labels over a loader (eval mode, no grad), in loader order."""
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for batch in loader:
            xa, xs, ta, ts, y = _batch_to_device(batch)
            logits, _, _, _ = model(xa, xs, ta, ts)
            probs.append(F.softmax(logits, dim=-1).cpu().numpy())
            labels.append(y.cpu().numpy())
    return np.concatenate(probs), np.concatenate(labels)


train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=NUM_WORKERS)
CLASS_WEIGHTS = train_class_weights(split_windows["train"]).to(device)
TEST_LABELS = np.array([w["label"] for w in split_windows["test"]], dtype=np.int64)   # test-window order, fixed


# Fixed probe batch for the lag-contribution check: timestamps of the first BATCH_SIZE training windows, read by
# index (no DataLoader, no RNG). The lag-bias term depends only on the timestamps.
LAG_PROBE_TAUS = tuple(torch.stack([train_ds[i][k] for i in range(min(BATCH_SIZE, len(train_ds)))])
                       for k in ("tau_a", "tau_s"))


def lag_bias_contribution(model):
    """Per LACA direction: max|beta * LN_grid(g(Δτ))|, the exact term PairwiseLACA adds to the attention scores
    (computed by the same lag_bias_term method, with the beta currently in effect), plus the raw std of g before
    the LayerNorm. g has no dropout or BatchNorm: this uses no random numbers and changes no state."""
    ta, ts = (t.to(device) for t in LAG_PROBE_TAUS)
    out = {}
    with torch.no_grad():
        for name, laca, tau_m, tau_n in (("laca_as", model.laca.laca_as, ta, ts), ("laca_sa", model.laca.laca_sa, ts, ta)):
            term = laca.lag_bias_term(tau_m, tau_n)
            contrib = laca.beta * term
            g_raw = laca.lag_bias((tau_m.unsqueeze(2) - tau_n.unsqueeze(1)).unsqueeze(-1)).squeeze(-1)
            out[name] = {"beta": laca.beta, "contrib_absmax": float(contrib.abs().max()),
                         "contrib_finite": bool(torch.isfinite(contrib).all()), "g_raw_std": float(g_raw.std())}
    return out


def g_final_layer_absmax(model):
    """max|weight| and max|bias| of g's final Linear(32, 1), per LACA direction."""
    return {name: (float(getattr(model.laca, name).lag_bias[-1].weight.detach().abs().max()),
                   float(getattr(model.laca, name).lag_bias[-1].bias.detach().abs().max()))
            for name in ("laca_as", "laca_sa")}


def check_warmup_history(beta_target, history, tag):
    """The beta actually in effect at each epoch must ramp 0 -> target over BETA_WARMUP_EPOCHS, then hold."""
    W = BETA_WARMUP_EPOCHS
    if beta_target == 0.0:
        assert all(b == 0.0 for b in history.values()), f"beta=0.0 run had nonzero beta: {history}"
        print(f"{tag} [WARM-UP CHECK] beta was 0.0 at all {len(history)} epochs -> the schedule is a no-op ✅")
        return
    assert abs(history[1] - beta_target / W) < 1e-12, f"epoch 1 beta {history[1]} != {beta_target / W}"
    assert all(history[e] < history[e + 1] for e in range(1, W)), f"beta not strictly increasing over epochs 1..{W}"
    assert abs(history[W] - beta_target) < 1e-12, f"epoch {W} beta {history[W]} != target {beta_target}"
    assert all(abs(history[e] - beta_target) < 1e-12 for e in history if e > W), "beta not held after the warm-up"
    print(f"{tag} [WARM-UP CHECK] ✅ ramping, not jumping: epoch 1 beta = {history[1]:.2f} | epoch {W} beta = "
          f"{history[W]:.2f} | epochs {W + 1}-{max(history)} beta = {history[W + 1]:.2f} (held) | full: "
          + ", ".join(f"{b:.1f}" for b in history.values()))


def train_one_run(seed: int, beta: float):
    """Train one model with `seed` applied to every RNG and target LACA scale `beta` (reached by the warm-up),
    evaluate it on the test set. Returns (trained model on CPU, result dict incl. test probabilities)."""
    t0 = time.time()
    tag = f"[seed={seed} beta={beta}]"
    set_all_seeds(seed)
    model = AeroSyncMamba(beta=beta).to(device)
    init_sha1 = hashlib.sha1(b"".join(v.detach().cpu().contiguous().numpy().tobytes()
                                      for v in model.state_dict().values())).hexdigest()
    rng_sha1 = hashlib.sha1(torch.get_rng_state().numpy().tobytes()).hexdigest()
    mamba_params = verify_architecture(model, beta=beta)
    assert model.beta_target == beta
    print(f"{tag} target beta read back from the constructed model: {model.beta_target} (reached via a "
          f"{BETA_WARMUP_EPOCHS}-epoch warm-up) | initial-weights fingerprint {init_sha1[:12]} | torch RNG after "
          f"construction {rng_sha1[:12]} | fusion {type(model.mamba_blocks[0]).__name__} ({mamba_params:,} params)")

    # --- ZERO-INIT check: before any training step ---
    zi = g_final_layer_absmax(model)
    assert all(w == 0.0 and b == 0.0 for w, b in zi.values()), f"g final layer is not zero-initialized: {zi}"
    lag0 = lag_bias_contribution(model)
    assert all(r["g_raw_std"] == 0.0 and r["contrib_absmax"] == 0.0 for r in lag0.values())
    print(f"{tag} [ZERO-INIT] ✅ before any training step: g final layer max|W| / max|b| = "
          + " | ".join(f"{n}: {w!r} / {b!r}" for n, (w, b) in zi.items())
          + " -> g(Δτ) = 0 for every Δτ; lag-bias term exactly 0 in both directions")
    if seed == 42:
        ok = rng_sha1.startswith(REFERENCE_SEED42_RNG_PREFIX)
        print(f"{tag} [RNG] torch RNG after construction {rng_sha1[:12]} vs v16 prefix {REFERENCE_SEED42_RNG_PREFIX}: "
              + ("match ✅ (zero-init and LayerNorm drew no random numbers; all other initial weights and the training "
                 "RNG stream are the same as v16)" if ok else "MISMATCH 🚨 (model construction consumes the RNG "
                 "differently from v16)"))

    criterion = AeroSyncLoss(CLASS_WEIGHTS)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)

    beta_history, lag_history = {}, {}
    best_val_f1, best_state, best_epoch, best_beta = 0.0, None, None, None
    for epoch in range(1, EPOCHS + 1):
        beta_now = beta_schedule(beta, epoch)
        model.set_beta(beta_now)
        applied = (model.laca.laca_as.beta, model.laca.laca_sa.beta)
        assert applied == (beta_now, beta_now), f"beta in effect {applied} != scheduled {beta_now}"
        beta_history[epoch] = applied[0]

        model.train()
        running = defaultdict(float)
        for batch in tqdm(train_loader, desc=f"seed {seed} beta {beta} epoch {epoch:02d}/{EPOCHS}", leave=False):
            xa, xs, ta, ts, y = _batch_to_device(batch, non_blocking=True)
            optimizer.zero_grad()
            logits, pooled, aux_logits, _ = model(xa, xs, ta, ts)
            loss, parts = criterion(logits, y, pooled, aux_logits)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
            optimizer.step()
            running["loss"] += loss.item()
            for k, v in parts.items():
                running[k] += v
        scheduler.step()

        lag = lag_bias_contribution(model)
        lag_max = max(r["contrib_absmax"] for r in lag.values())
        lag_history[epoch] = lag_max
        if beta == 0.0:
            assert all(r["contrib_finite"] and r["contrib_absmax"] == 0.0 for r in lag.values()), (
                f"ABLATION LEAK at epoch {epoch}: {lag}")

        val_probs, val_labels = predict_probs(model, val_loader)
        val_f1 = compute_metrics(val_labels, val_probs)["macro_f1"]
        improved = val_f1 > best_val_f1
        if improved:
            best_val_f1, best_state, best_epoch, best_beta = val_f1, copy.deepcopy(model.state_dict()), epoch, beta_now
        n = len(train_loader)
        print(f"{tag} epoch {epoch:02d}/{EPOCHS} | beta {beta_now:.2f} | max|beta*LN(g)| {lag_max:.4f}"
              f"{' (exactly 0 ✅)' if beta == 0.0 else ''} | raw g std {max(r['g_raw_std'] for r in lag.values()):.2e}"
              f" | train loss {running['loss'] / n:.4f} (cls {running['loss_cls'] / n:.4f}, align "
              f"{running['loss_align'] / n:.4f}, cons {running['loss_cons'] / n:.4f}) | val macro-F1 {val_f1:.4f}"
              f"{'  * best' if improved else ''}")
        if beta > 0.0 and epoch in (1, BETA_WARMUP_EPOCHS, BETA_WARMUP_EPOCHS + 1):
            print(f"{tag} [WARM-UP] epoch {epoch}: beta in effect = {beta_now:.2f}"
                  + (" (ramping)" if epoch < BETA_WARMUP_EPOCHS else " (target reached)" if epoch == BETA_WARMUP_EPOCHS
                     else " (held at target)"))

    check_warmup_history(beta, beta_history, tag)
    if beta == 0.0:
        zi_end = g_final_layer_absmax(model)
        print(f"{tag} [ABLATION] ✅ clean at EVERY epoch: beta * LN(g(Δτ)) was exactly 0.0 in both directions after "
              f"each of the {EPOCHS} epochs (g's final layer is still exactly zero: {zi_end})")
    lag_final = lag_history[EPOCHS]

    model.load_state_dict(best_state)
    model.set_beta(best_beta)   # evaluate the checkpoint with the beta it was validated with
    lag_best = max(r["contrib_absmax"] for r in lag_bias_contribution(model).values())
    if beta > 0.0:
        print(f"{tag} [LAG CONTRIBUTION] max|beta*LN(g)|: final epoch {lag_final:.4f} | best checkpoint (epoch "
              f"{best_epoch}, beta {best_beta:.2f}) {lag_best:.4f}")
        if best_beta != beta:
            print(f"{tag} ⚠️  best checkpoint is from the warm-up (epoch {best_epoch}); tested with beta {best_beta:.2f}")
    test_probs, test_labels = predict_probs(model, test_loader)
    metrics = compute_metrics(test_labels, test_probs)
    run_name = f"beta{beta}_seed{seed}"
    torch.save(best_state, CHECKPOINT_DIR / f"{run_name}_best.pt")
    np.save(PROBS_DIR / f"{run_name}_test_probs.npy", test_probs)
    np.save(PROBS_DIR / f"{run_name}_test_labels.npy", test_labels)

    result = {
        "beta": beta,
        "seed": seed,
        **metrics,
        "best_val_macro_f1": round(best_val_f1, 6),
        "best_epoch": best_epoch,
        "beta_at_best": best_beta,
        "n_test_windows": len(test_labels),
        "init_sha1": init_sha1[:12],
        "rng_sha1": rng_sha1[:12],
        "lag_contrib_final_epoch": lag_final,
        "lag_contrib_best": lag_best,
        "lag_zero_every_epoch": all(v == 0.0 for v in lag_history.values()),
        "beta_epoch1": beta_history[1],
        "beta_epoch5": beta_history[BETA_WARMUP_EPOCHS],
        "beta_epoch6": beta_history[BETA_WARMUP_EPOCHS + 1],
        "train_seconds": round(time.time() - t0, 1),
        "test_probs": test_probs,
        "test_labels": test_labels,
    }
    return model.cpu(), result

## 6. Paired training driver: 10 runs
For each seed in 42, 123, 2024, 7, 99, the driver runs β = 0.0 and then β = 1.0 back to back. After every run it appends a row to `/kaggle/working/beta_stabilized_paired_results.csv`, frees the GPU, and prints a progress line.

- **Seed 42 / β = 1.0** is compared with the v16 baseline for information only. The architecture changed, so a different result is expected.
- **Each β = 0.0 run** is checked for exact reproduction of v17's β = 0.0 run, if v17's CSV is available. At β = 0 none of the stabilization changes reach the output.
- **Each finished pair** is checked for identical initial weights and an identical RNG stream.

In [7]:
# ==============================================================================
# PAIRED DRIVER — for seed in SEEDS: beta=0.0 then beta=1.0. After every run:
# metrics, CSV row, GPU freed, progress. Seed 42 / beta=1.0 vs v16: INFORMATIONAL
# (expected divergence). beta=0.0 vs v17: exact-reproduction control (if v17's
# CSV is available). Every finished pair: identical initial weights + RNG stream.
# ==============================================================================
import glob

import pandas as pd

RESULTS_CSV_FIELDS = ["run", "beta", "seed", "accuracy", "macro_f1", "balanced_accuracy", "macro_auroc",
                      "best_val_macro_f1", "best_epoch", "beta_at_best", "n_test_windows", "init_sha1", "rng_sha1",
                      "lag_contrib_final_epoch", "lag_contrib_best", "lag_zero_every_epoch", "beta_epoch1",
                      "beta_epoch5", "beta_epoch6", "train_seconds", "finished_at"]
METRICS = ["accuracy", "macro_f1", "balanced_accuracy", "macro_auroc"]

# --- optional: v17's per-run results (un-stabilized) ---
V17_DF = None
for _pattern in V17_RESULTS_CSV_CANDIDATES:
    _hits = sorted(glob.glob(_pattern, recursive=True))
    if _hits:
        V17_DF = pd.read_csv(_hits[0])
        print(f"[v17] loaded {_hits[0]} ({len(V17_DF)} rows) -> beta=0 exact-reproduction control ENABLED")
        break
if V17_DF is None:
    print("[v17] v17 CSV not found -> beta=0 exact-reproduction control SKIPPED; the v17 comparison in Section 7 uses "
          "the reported numbers (V17_PAIRED). Attach beta_paired_replication_results.csv as a Kaggle input to enable it.")


def v17_row(seed, beta):
    if V17_DF is None:
        return None
    rows = V17_DF[(V17_DF["seed"] == seed) & (V17_DF["beta"] == beta)]
    return rows.iloc[-1] if len(rows) else None


if RESULTS_CSV.exists():
    RESULTS_CSV.rename(RESULTS_CSV.with_suffix(f".{datetime.now():%Y%m%d_%H%M%S}.bak.csv"))


def report_seed42_vs_v16(r):
    print("\n" + "#" * 100 + "\n#  seed 42 / beta=1.0 vs the v16 baseline — INFORMATIONAL, NOT A REPRODUCTION CHECK\n" + "#" * 100)
    deltas = {}
    for k, ref in V16_BASELINE_SEED42.items():
        deltas[k] = r[k] - ref
        print(f"#  {k:<18} v16 {ref:.4f} | v18 {r[k]:.4f} | delta {deltas[k]:+.4f}")
    print("#  EXPECTED DIVERGENCE: v18 intentionally changes the LACA lag-bias mechanism (LayerNorm on g's output,")
    print("#  zero-initialized g final layer, 5-epoch beta warm-up). A different result from v16's 0.9080 / 0.9048 is")
    print("#  the expected consequence of that architecture change, NOT a bug or a failed reproduction. What is still")
    print("#  checked: data fingerprints, the post-construction RNG state (must match v16), and the beta=0 arm vs v17.")
    if all(abs(d) <= SAME_SEED_TOLERANCE for d in deltas.values()):
        print("#  ⚠️  every metric matches v16 within ±0.0005 — suspicious: check the stabilization is actually active.")
    print("#" * 100)


V17_CONTROL = {}   # seed -> True/False (beta=0 reproduces v17 exactly)


def check_beta0_vs_v17(seed, r):
    row = v17_row(seed, 0.0)
    if row is None:
        print(f"[v17 CONTROL seed={seed}] skipped (no v17 row)")
        return
    worst = max(abs(r[k] - float(row[k])) for k in METRICS)
    V17_CONTROL[seed] = worst <= SAME_SEED_TOLERANCE
    if V17_CONTROL[seed]:
        print(f"[v17 CONTROL seed={seed}] ✅ beta=0.0 reproduces v17 (max |delta| {worst:.4f}): at beta=0 the "
              f"stabilization never reaches the output, so this arm is an unchanged control")
    else:
        print("🚨" * 45)
        print(f"🚨  seed {seed} beta=0.0 does NOT reproduce v17 (max |delta| {worst:.4f}; macro-F1 {r['macro_f1']:.4f} vs "
              f"{float(row['macro_f1']):.4f}). At beta=0 nothing that reaches the output changed, so something else")
        print("🚨  differs (environment / library / data) — the v17-vs-v18 comparison is confounded for this seed.")
        print("🚨" * 45)


run_results = {}
t_driver = time.time()
for i, (seed, beta) in enumerate(RUN_ORDER, start=1):
    print("\n" + "=" * 100 + f"\nRUN {i}/{len(RUN_ORDER)} — seed={seed}, beta={beta}"
          + ("  (lag-bias ABLATION)" if beta == 0.0 else "  (stabilized lag bias, warm-up to 1.0)") + "\n" + "=" * 100)
    model, result = train_one_run(seed, beta)
    run_results[(seed, beta)] = result
    print(f"[seed={seed} beta={beta}] TEST: accuracy {result['accuracy']:.4f} | macro-F1 {result['macro_f1']:.4f} | "
          f"balanced accuracy {result['balanced_accuracy']:.4f} | macro-AUROC {result['macro_auroc']:.4f} | best epoch "
          f"{result['best_epoch']} | {result['train_seconds']:.0f}s")

    write_header = not RESULTS_CSV.exists()
    with open(RESULTS_CSV, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=RESULTS_CSV_FIELDS)
        if write_header:
            writer.writeheader()
        writer.writerow({"run": i, **{k: result[k] for k in RESULTS_CSV_FIELDS if k in result},
                         "finished_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")})

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if beta == 0.0:
        check_beta0_vs_v17(seed, result)
    if seed == 42 and beta == 1.0:
        report_seed42_vs_v16(result)
    if beta > 0.0:
        lags = {s: run_results[(s, 1.0)]["lag_contrib_final_epoch"] for s in SEEDS if (s, 1.0) in run_results}
        print(f"[LAG SPREAD] final-epoch max|beta*LN(g)| for beta=1.0 runs so far: "
              + ", ".join(f"seed {s}: {v:.4f}" for s, v in lags.items())
              + f" | range {min(lags.values()):.4f}-{max(lags.values()):.4f} (v17 un-stabilized: "
              f"{V17_PAIRED['lag_min']:.2f}-{V17_PAIRED['lag_max']:.2f})")

    # --- pair complete? only beta may differ between its two runs ---
    if all((seed, b) in run_results for b in BETA_VALUES):
        r0, r1 = run_results[(seed, 0.0)], run_results[(seed, 1.0)]
        assert r0["init_sha1"] == r1["init_sha1"], f"seed {seed}: beta=0/1 initial weights differ — not a paired design"
        assert r0["rng_sha1"] == r1["rng_sha1"], f"seed {seed}: beta=0/1 RNG streams differ — not a paired design"
        assert np.array_equal(r0["test_labels"], r1["test_labels"]), f"seed {seed}: test windows differ"
        print(f"[PAIR seed={seed}] ✅ identical initial weights ({r0['init_sha1']}) and RNG stream ({r0['rng_sha1']}) "
              f"for beta=0.0 and beta=1.0 -> only beta differs | macro-F1 beta=0 {r0['macro_f1']:.4f}, beta=1 "
              f"{r1['macro_f1']:.4f}, paired diff (1 - 0) {r1['macro_f1'] - r0['macro_f1']:+.4f}")

    n_pairs = sum(all((s, b) in run_results for b in BETA_VALUES) for s in SEEDS)
    elapsed = time.time() - t_driver
    print(f"\n>>> {i}/{len(RUN_ORDER)} complete: seed={seed}, beta={beta} finished | complete pairs {n_pairs}/{len(SEEDS)} | "
          f"elapsed {elapsed / 60:.1f} min | est. remaining {elapsed / i * (len(RUN_ORDER) - i) / 60:.1f} min")

init_prints = {s: run_results[(s, 1.0)]["init_sha1"] for s in SEEDS}
rng_prints = {s: run_results[(s, 1.0)]["rng_sha1"] for s in SEEDS}
assert len(set(init_prints.values())) == len(SEEDS), f"two seeds produced identical initial weights: {init_prints}"
assert len(set(rng_prints.values())) == len(SEEDS), f"two seeds produced identical RNG streams: {rng_prints}"
print(f"\n[SEED CHECK] {len(SEEDS)} seeds -> distinct initial weights / RNG streams; each shared by its beta=0/1 pair: "
      + ", ".join(f"{s}: {init_prints[s]}/{rng_prints[s]}" for s in SEEDS))
print(f"All {len(RUN_ORDER)} runs finished in {(time.time() - t_driver) / 60:.1f} min. Results: {RESULTS_CSV}")

[v17] v17 CSV not found -> beta=0 exact-reproduction control SKIPPED; the v17 comparison in Section 7 uses the reported numbers (V17_PAIRED). Attach beta_paired_replication_results.csv as a Kaggle input to enable it.

RUN 1/10 — seed=42, beta=0.0  (lag-bias ABLATION)
[seed=42 beta=0.0] target beta read back from the constructed model: 0.0 (reached via a 5-epoch warm-up) | initial-weights fingerprint 5850841e699f | torch RNG after construction 5641b401720d | fusion MambaBlock (874,496 params)
[seed=42 beta=0.0] [ZERO-INIT] ✅ before any training step: g final layer max|W| / max|b| = laca_as: 0.0 / 0.0 | laca_sa: 0.0 / 0.0 -> g(Δτ) = 0 for every Δτ; lag-bias term exactly 0 in both directions
[seed=42 beta=0.0] [RNG] torch RNG after construction 5641b401720d vs v16 prefix 5641b401720d: match ✅ (zero-init and LayerNorm drew no random numbers; all other initial weights and the training RNG stream are the same as v16)


seed 42 beta 0.0 epoch 01/20:   0%|          | 0/155 [00:00<?, ?it/s]

[seed=42 beta=0.0] epoch 01/20 | beta 0.00 | max|beta*LN(g)| 0.0000 (exactly 0 ✅) | raw g std 0.00e+00 | train loss 1.4174 (cls 1.0707, align 3.1025, cons 0.3650) | val macro-F1 0.8261  * best


seed 42 beta 0.0 epoch 02/20:   0%|          | 0/155 [00:00<?, ?it/s]

[seed=42 beta=0.0] epoch 02/20 | beta 0.00 | max|beta*LN(g)| 0.0000 (exactly 0 ✅) | raw g std 0.00e+00 | train loss 0.9259 (cls 0.5977, align 2.6650, cons 0.6171) | val macro-F1 0.6814


...

seed 99 beta 1.0 epoch 19/20:   0%|          | 0/155 [00:00<?, ?it/s]

[seed=99 beta=1.0] epoch 19/20 | beta 1.00 | max|beta*LN(g)| 2.1200 | raw g std 5.06e-03 | train loss 0.3028 (cls 0.1261, align 1.3305, cons 0.4368) | val macro-F1 0.8823  * best


seed 99 beta 1.0 epoch 20/20:   0%|          | 0/155 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c502e280360>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c502e280360>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

[seed=99 beta=1.0] epoch 20/20 | beta 1.00 | max|beta*LN(g)| 2.1114 | raw g std 4.94e-03 | train loss 0.3024 (cls 0.1267, align 1.3208, cons 0.4361) | val macro-F1 0.8857  * best
[seed=99 beta=1.0] [WARM-UP CHECK] ✅ ramping, not jumping: epoch 1 beta = 0.20 | epoch 5 beta = 1.00 | epochs 6-20 beta = 1.00 (held) | full: 0.2, 0.4, 0.6, 0.8, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0
[seed=99 beta=1.0] [LAG CONTRIBUTION] max|beta*LN(g)|: final epoch 2.1114 | best checkpoint (epoch 20, beta 1.00) 2.1114
[seed=99 beta=1.0] TEST: accuracy 0.8761 | macro-F1 0.8712 | balanced accuracy 0.8687 | macro-AUROC 0.9845 | best epoch 20 | 244s
[LAG SPREAD] final-epoch max|beta*LN(g)| for beta=1.0 runs so far: seed 42: 1.7811, seed 123: 1.9592, seed 2024: 2.0692, seed 7: 1.2868, seed 99: 2.1114 | range 1.2868-2.1114 (v17 un-stabilized: 0.23-0.68)
[PAIR seed=99] ✅ identical initial weights (e40c78169f57) and RNG stream (81dcd4638a88) for beta=0.0 and beta=1.0 -> only b

## 7. Paired statistics, v17 side-by-side, and four separate answers
This section reads `beta_stabilized_paired_results.csv` and uses complete pairs only. It computes the same statistics as v17: a per-seed paired table for all four metrics, then for macro-F1 the mean paired difference, the std of the paired differences, the win count, a **paired t-test (primary)** and a Wilcoxon signed-rank test (secondary).

**Why the t-test is primary.** With 5 pairs, the exact two-sided Wilcoxon p-value can never go below 2/2⁵ = 0.0625, so it can never reach 0.05. The t-test uses the size of each difference, but it assumes the differences are roughly normal, which can't be checked with 5 pairs. So the Wilcoxon test is reported as a robustness check.

**Comparison with v17.** A side-by-side table against v17's un-stabilized result, then four questions answered separately: (a) did the spread of the lag contribution across seeds narrow, (b) did the mean paired difference improve, (c) did the std of the paired differences shrink, (d) did the win count change.

**About (a).** Stabilization changes the units of the lag contribution. After the LayerNorm, max|β·LN(g)| is β times the largest z-score in the grid, so it is not on v17's raw scale. (a) is therefore decided with the scale-free max/min ratio (v17: 0.68 / 0.23 ≈ 2.96). The raw ranges are printed as well.

**Final epoch vs best checkpoint.** v17's CSV recorded this quantity at the best checkpoint. This notebook reports both the final epoch and the best checkpoint, and uses the best checkpoint for the like-for-like comparison.

In [8]:
# ==============================================================================
# PAIRED ANALYSIS (same statistics as v17) + v17 SIDE-BY-SIDE + (a)-(d)
# ==============================================================================
from scipy import stats

df = pd.read_csv(RESULTS_CSV)
complete_seeds = [s for s in SEEDS if set(df.loc[df["seed"] == s, "beta"]) >= set(BETA_VALUES)]
n = len(complete_seeds)
print(f"Complete (beta=0, beta=1) pairs: {n}/{len(SEEDS)} -> seeds {complete_seeds}")
if n < len(SEEDS):
    print(f"⚠️  PARTIAL RESULT: only {n} of {len(SEEDS)} pairs finished — the statistics below use those pairs only.")
assert n >= 2, "need at least 2 complete pairs for any paired statistic"
pivot = {m: df.pivot_table(index="seed", columns="beta", values=m).loc[complete_seeds]
         for m in METRICS + ["lag_contrib_final_epoch", "lag_contrib_best"]}


def paired_stats(b1, b0):
    d = np.asarray(b1, dtype=float) - np.asarray(b0, dtype=float)
    k = len(d)
    out = {"d": d, "n": k, "mean": float(d.mean()), "sd": float(d.std(ddof=1)), "wins": int((d > 0).sum()),
           "losses": int((d < 0).sum()), "ties": int((d == 0).sum())}
    out["se"] = out["sd"] / math.sqrt(k)
    tr = stats.ttest_rel(b1, b0)
    out["t"], out["p_t"] = float(tr.statistic), float(tr.pvalue)
    tc = float(stats.t.ppf(0.975, df=k - 1))
    out["ci"] = (out["mean"] - tc * out["se"], out["mean"] + tc * out["se"])
    try:
        out["p_w"] = float(stats.wilcoxon(d, alternative="two-sided", method="exact").pvalue)
    except Exception:
        out["p_w"] = float("nan")
    return out


# --- 1. per-seed paired tables, all 4 metrics ---
for m in METRICS:
    p = pivot[m]
    d = p[1.0] - p[0.0]
    print(f"\n{'PRIMARY — ' if m == 'macro_f1' else ''}{m}")
    print(f"  {'seed':>6} {'beta=0.0':>10} {'beta=1.0':>10} {'diff (1-0)':>11}  favors")
    for s in complete_seeds:
        fav = "beta=1" if d[s] > 0 else ("beta=0" if d[s] < 0 else "tie")
        print(f"  {s:>6} {p.loc[s, 0.0]:>10.4f} {p.loc[s, 1.0]:>10.4f} {d[s]:>+11.4f}  {fav}")
    print(f"  {'mean':>6} {p[0.0].mean():>10.4f} {p[1.0].mean():>10.4f} {d.mean():>+11.4f}  "
          f"(std of paired diffs {d.std(ddof=1):.4f}; beta=1 wins {int((d > 0).sum())}/{n})")

# --- 2. paired statistics on macro-F1 ---
P = paired_stats(pivot["macro_f1"][1.0].to_numpy(), pivot["macro_f1"][0.0].to_numpy())
mean_d, sd_d, wins, p_t = P["mean"], P["sd"], P["wins"], P["p_t"]
print("\n" + "=" * 100)
print(f"PAIRED STATISTICS (v18, stabilized) — test macro-F1, beta=1.0 minus beta=0.0, {n} matched seeds")
print("=" * 100)
print(f"  paired differences          : " + ", ".join(f"{s}: {x:+.4f}" for s, x in zip(complete_seeds, P["d"])))
print(f"  mean paired difference      : {mean_d:+.4f}")
print(f"  std of paired differences   : {sd_d:.4f}   (sample std, ddof=1)")
print(f"  95% CI of mean difference   : [{P['ci'][0]:+.4f}, {P['ci'][1]:+.4f}]")
print(f"  win count                   : beta=1 better on {wins}/{n} seeds (beta=0 better on {P['losses']}, ties {P['ties']})")
print(f"  paired t-test (PRIMARY)     : t = {P['t']:+.3f}, df = {n - 1}, two-sided p = {p_t:.4f}")
print(f"  Wilcoxon signed-rank (exact): two-sided p = {P['p_w']:.4f}   (smallest achievable with n={n}: {2 / 2 ** n:.4f})")

# --- 3. lag-contribution spread (beta=1.0 runs) ---
lag_best = pivot["lag_contrib_best"][1.0]
lag_final = pivot["lag_contrib_final_epoch"][1.0]
print("\nLAG-BIAS CONTRIBUTION max|beta*LN(g(Δτ))|, beta=1.0 runs:")
for s in complete_seeds:
    print(f"  seed {s:<5} final epoch {lag_final[s]:.4f} | best checkpoint {lag_best[s]:.4f}")


def spread(v):
    v = np.asarray(v, dtype=float)
    return {"min": float(v.min()), "max": float(v.max()), "ratio": float(v.max() / v.min()) if v.min() > 0 else float("inf"),
            "cv": float(v.std(ddof=1) / v.mean()) if v.mean() > 0 else float("nan")}


SP_best, SP_final = spread(lag_best), spread(lag_final)

# --- 4. v17 reference: reported numbers, recomputed from its CSV when available ---
V17 = dict(V17_PAIRED)
V17["ratio"] = V17["lag_max"] / V17["lag_min"]
V17_P = None
if V17_DF is not None:
    _v17_seeds = [s for s in complete_seeds if v17_row(s, 0.0) is not None and v17_row(s, 1.0) is not None]
    if len(_v17_seeds) >= 2:
        _b1 = np.array([float(v17_row(s, 1.0)["macro_f1"]) for s in _v17_seeds])
        _b0 = np.array([float(v17_row(s, 0.0)["macro_f1"]) for s in _v17_seeds])
        V17_P = paired_stats(_b1, _b0)
        print(f"\n[v17 CSV] recomputed on seeds {_v17_seeds}: mean {V17_P['mean']:+.4f}, std {V17_P['sd']:.4f}, wins "
              f"{V17_P['wins']}/{V17_P['n']}, p {V17_P['p_t']:.4f} (reported: +0.0008, 0.0020, 3/5)")
        V17.update(mean_diff=V17_P["mean"], sd_diff=V17_P["sd"], wins=V17_P["wins"], n=V17_P["n"], p_t=V17_P["p_t"])
        if "lag_contrib_absmax_best" in V17_DF.columns:
            _lags = [float(v17_row(s, 1.0)["lag_contrib_absmax_best"]) for s in _v17_seeds]
            V17.update(lag_min=min(_lags), lag_max=max(_lags), ratio=max(_lags) / min(_lags))

# --- 5. side-by-side ---
print("\n" + "=" * 100)
print("SIDE-BY-SIDE — v17 (un-stabilized) vs v18 (stabilized), macro-F1 paired diff = beta=1 minus beta=0")
print("=" * 100)
print(f"  {'':<38} {'v17 un-stabilized':>22} {'v18 stabilized':>22}")
print(f"  {'mean paired difference':<38} {V17['mean_diff']:>+22.4f} {mean_d:>+22.4f}")
print(f"  {'std of paired differences':<38} {V17['sd_diff']:>22.4f} {sd_d:>22.4f}")
print(f"  {'win count (beta=1 > beta=0)':<38} {str(V17['wins']) + '/' + str(V17['n']):>22} {str(wins) + '/' + str(n):>22}")
print(f"  {'paired t-test p':<38} {(format(V17['p_t'], '.4f') if 'p_t' in V17 else 'n/a'):>22} {p_t:>22.4f}")
print(f"  {'lag contribution range (best ckpt)':<38} {format(V17['lag_min'], '.2f') + '-' + format(V17['lag_max'], '.2f'):>22} "
      f"{format(SP_best['min'], '.3f') + '-' + format(SP_best['max'], '.3f'):>22}")
print(f"  {'lag contribution max/min (scale-free)':<38} {V17['ratio']:>22.2f} {SP_best['ratio']:>22.2f}")
print(f"  {'lag range at FINAL epoch (v18 only)':<38} {'-':>22} "
      f"{format(SP_final['min'], '.3f') + '-' + format(SP_final['max'], '.3f') + ' (x' + format(SP_final['ratio'], '.2f') + ')':>22}")

# --- 6. the four questions, answered separately ---
print("\n" + "#" * 100)
print("#  FOUR SEPARATE QUESTIONS")
print("#" * 100)

# (a) spread
narrowed = SP_best["ratio"] < V17["ratio"]
print(f"#  (a) Did stabilization NARROW the lag-contribution spread across seeds?  {'YES' if narrowed else 'NO'}")
print(f"#      max/min across seeds: v17 {V17['ratio']:.2f} -> v18 {SP_best['ratio']:.2f} (best checkpoint; final epoch "
      f"{SP_final['ratio']:.2f}"
      + ("" if (SP_final["ratio"] < V17["ratio"]) == narrowed else ", which gives the OPPOSITE answer") + ").")
print("#      Caveat: after the LayerNorm, beta*LN(g) has unit std by construction (for beta=1), so part of any")
print("#      narrowing is mechanical. What still varies across seeds is the SHAPE of g over Δτ (its largest z-score).")

# (b) mean
improved = mean_d > V17["mean_diff"]
if V17_P is not None:
    _x = pivot["macro_f1"][1.0].loc[_v17_seeds].to_numpy() - np.array([float(v17_row(s, 1.0)["macro_f1"]) for s in _v17_seeds])
    _xs = stats.ttest_1samp(_x, 0.0) if len(_x) >= 2 else None
    b_detail = (f"per-seed change in the beta=1 arm (v18 - v17; the beta=0 arm is an unchanged control): mean "
                f"{_x.mean():+.4f}, paired t-test p = {_xs.pvalue:.4f}")
    b_clear = _xs is not None and _xs.pvalue < PAIRED_ALPHA
else:
    _se = math.sqrt((V17["sd_diff"] / math.sqrt(V17["n"])) ** 2 + P["se"] ** 2)
    b_detail = (f"change {mean_d - V17['mean_diff']:+.4f} vs approx. combined standard error {_se:.4f} (2 x SE = "
                f"{2 * _se:.4f}; v17 CSV not available for the exact per-seed test)")
    b_clear = abs(mean_d - V17["mean_diff"]) > 2 * _se
print(f"#  (b) Did stabilization IMPROVE the mean paired difference?  {'YES' if improved else 'NO'} — "
      f"{V17['mean_diff']:+.4f} -> {mean_d:+.4f} "
      + ("(a clear change, beyond noise)" if b_clear else "(the change is within noise)"))
print(f"#      {b_detail}")

# (c) std
reduced = sd_d < V17["sd_diff"]
_ratio_needed = math.sqrt(1.0 / float(stats.f.ppf(1 - PAIRED_ALPHA, n - 1, V17["n"] - 1)))
decisive = sd_d / V17["sd_diff"] < _ratio_needed if V17["sd_diff"] > 0 else False
print(f"#  (c) Did stabilization REDUCE the std of the paired differences (more CONSISTENT effect)?  "
      f"{'YES' if reduced else 'NO'} — {V17['sd_diff']:.4f} -> {sd_d:.4f} (ratio {sd_d / V17['sd_diff']:.2f})")
print(f"#      With {n} and {V17['n']} pairs, a std ratio below about {_ratio_needed:.2f} is needed before the reduction "
      f"is more than noise (approximate F-test; the two designs share seeds and the beta=0 arm): "
      + ("the reduction is DECISIVE." if decisive else "NOT decisive."))

# (d) wins
wins_change = "INCREASED" if wins * V17["n"] > V17["wins"] * n else ("DECREASED" if wins * V17["n"] < V17["wins"] * n else "UNCHANGED")
print(f"#  (d) Did the win count change?  {wins_change} — v17 {V17['wins']}/{V17['n']} -> v18 {wins}/{n} seeds favor beta=1.0")

# --- 7. verdict on beta for the stabilized architecture (same rule as v17) ---
exceeds_noise = abs(mean_d) > sd_d
significant = p_t < PAIRED_ALPHA
beta1_consistent = mean_d > 0 and wins >= n - 1
beta0_consistent = mean_d < 0 and P["losses"] >= n - 1
if beta1_consistent and (exceeds_noise or significant):
    VERDICT = "WELL-SUPPORTED"
elif beta0_consistent and (exceeds_noise or significant):
    VERDICT = "REVERSED"
else:
    VERDICT = "NOT WELL-SUPPORTED"
print("#" * 100)
print(f"#  VERDICT (same rule as v17): on the STABILIZED architecture, beta=1.0's advantage over beta=0.0 is {VERDICT}")
print(f"#  wins {wins}/{n} (need >= {n - 1}) | |mean| {abs(mean_d):.4f} vs std {sd_d:.4f} | paired t-test p = {p_t:.4f}")
if n < len(SEEDS):
    print(f"#  ⚠️  based on {n}/{len(SEEDS)} pairs only.")
print("#" * 100)

Complete (beta=0, beta=1) pairs: 5/5 -> seeds [42, 123, 2024, 7, 99]

accuracy
    seed   beta=0.0   beta=1.0  diff (1-0)  favors
      42     0.9088     0.8814     -0.0274  beta=0
     123     0.8876     0.8858     -0.0018  beta=0
    2024     0.8805     0.8699     -0.0106  beta=0
       7     0.9062     0.8938     -0.0124  beta=0
      99     0.8690     0.8761     +0.0071  beta=1
    mean     0.8904     0.8814     -0.0090  (std of paired diffs 0.0129; beta=1 wins 1/5)

PRIMARY — macro_f1
    seed   beta=0.0   beta=1.0  diff (1-0)  favors
      42     0.9057     0.8768     -0.0290  beta=0
     123     0.8827     0.8812     -0.0015  beta=0
    2024     0.8764     0.8652     -0.0112  beta=0
       7     0.9019     0.8892     -0.0127  beta=0
      99     0.8638     0.8712     +0.0074  beta=1
    mean     0.8861     0.8767     -0.0094  (std of paired diffs 0.0136; beta=1 wins 1/5)

balanced_accuracy
    seed   beta=0.0   beta=1.0  diff (1-0)  favors
      42     0.9030     0.8749     -0.0

## 8. Final validation
A recap of every verification hook: data and RNG diagnostics, zero-init, the warm-up schedule, a clean β = 0 ablation at every epoch, the β = 0 control against v17, pair integrity, the intended divergence from v16, and the verdict.

In [9]:
# ==============================================================================
# FINAL VALIDATION — every verification hook in one place
# ==============================================================================
print("=" * 100)
print("FINAL VALIDATION")
print("=" * 100)
r42 = run_results[(42, 1.0)]
data_checks = {
    "strongest-64 bin indices": SELECTED_BINS.tolist() == REFERENCE_DATA["bin_indices"],
    **{f"{s} windows / sensor values / timestamps / class counts": DATA_FINGERPRINT[s] ==
       REFERENCE_DATA["sensor_fingerprint"][s] for s in ("train", "val", "test")},
    "seed-42 torch RNG after construction == v16 (no extra draws)": r42["rng_sha1"].startswith(REFERENCE_SEED42_RNG_PREFIX),
    "fusion backbone is real mamba_ssm": MAMBA_CLASS is not None,
}
for name, ok in data_checks.items():
    print(f"  {name:<62}: {'match' if ok else 'MISMATCH'}")

b1_runs = [r for (s, b), r in run_results.items() if b == 1.0]
b0_runs = [r for (s, b), r in run_results.items() if b == 0.0]
warmup_ok = all(abs(r["beta_epoch1"] - 1.0 / BETA_WARMUP_EPOCHS) < 1e-12 and r["beta_epoch5"] == 1.0 and r["beta_epoch6"] == 1.0
                for r in b1_runs)
ablation_ok = all(r["lag_zero_every_epoch"] for r in b0_runs)
pairs_ok = all(run_results[(s, 0.0)]["init_sha1"] == run_results[(s, 1.0)]["init_sha1"] and
               run_results[(s, 0.0)]["rng_sha1"] == run_results[(s, 1.0)]["rng_sha1"] for s in complete_seeds)

print("\n" + "=" * 100)
print(f"  zero-init (g final layer W, b == 0 before training) : ✅ asserted for all {len(run_results)} runs")
print(f"  beta warm-up (ep1 0.2 -> ep5 1.0 -> ep6+ 1.0)        : {'✅' if warmup_ok else '❌'} ({len(b1_runs)} beta=1.0 runs)")
print(f"  beta=0 lag term exactly 0 at EVERY epoch             : {'✅' if ablation_ok else '❌'} ({len(b0_runs)} beta=0.0 runs x {EPOCHS} epochs)")
print(f"  beta=0 arm reproduces v17 exactly                    : "
      + (("✅ all seeds" if all(V17_CONTROL.values()) else "❌ " + str({s: v for s, v in V17_CONTROL.items() if not v}))
         if V17_CONTROL else "skipped (v17 CSV not available)"))
print(f"  pairs share initial weights + RNG stream             : {'✅' if pairs_ok else '❌'} ({n} complete pairs)")
print(f"  seed 42 / beta=1.0 vs v16 baseline                   : macro-F1 {r42['macro_f1']:.4f} vs "
      f"{V16_BASELINE_SEED42['macro_f1']:.4f} (delta {r42['macro_f1'] - V16_BASELINE_SEED42['macro_f1']:+.4f}) — "
      f"EXPECTED divergence (architecture changed), not a failed reproduction")
print(f"  v17 -> v18: (a) spread narrowed {'YES' if narrowed else 'NO'} | (b) mean improved {'YES' if improved else 'NO'} | "
      f"(c) paired std reduced {'YES' if reduced else 'NO'} | (d) wins {wins_change} ({V17['wins']}/{V17['n']} -> {wins}/{n})")
print(f"  VERDICT (stabilized)                                 : beta=1.0 advantage {VERDICT} — mean diff {mean_d:+.4f}, "
      f"std {sd_d:.4f}, wins {wins}/{n}, p = {p_t:.4f}")
print("=" * 100)
if not all(data_checks.values()):
    print("🚨 a data / RNG diagnostic is a MISMATCH — the data or model construction differs from v16/v17 beyond the "
          "intended lag-bias change; the comparison with v17 is not clean.")

FINAL VALIDATION
  strongest-64 bin indices                                      : match
  train windows / sensor values / timestamps / class counts     : match
  val windows / sensor values / timestamps / class counts       : match
  test windows / sensor values / timestamps / class counts      : match
  seed-42 torch RNG after construction == v16 (no extra draws)  : match
  fusion backbone is real mamba_ssm                             : match

  zero-init (g final layer W, b == 0 before training) : ✅ asserted for all 10 runs
  beta warm-up (ep1 0.2 -> ep5 1.0 -> ep6+ 1.0)        : ✅ (5 beta=1.0 runs)
  beta=0 lag term exactly 0 at EVERY epoch             : ✅ (5 beta=0.0 runs x 20 epochs)
  beta=0 arm reproduces v17 exactly                    : skipped (v17 CSV not available)
  pairs share initial weights + RNG stream             : ✅ (5 complete pairs)
  seed 42 / beta=1.0 vs v16 baseline                   : macro-F1 0.8768 vs 0.9048 (delta -0.0280) — EXPECTED divergence (architecture